# 🚢 Titanic Survival Prediction
### Binary Classification using Logistic Regression & KNN
> Predicting whether a passenger survived the Titanic disaster based on features like age, sex, class, and more.

## 1. 📦 Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve
)

## 2. 📂 Load Dataset

In [ ]:
titanic_df = pd.read_csv("Titanic-Dataset.csv")  # Update path if needed
titanic_df.head()

In [ ]:
print("DataFrame shape:", titanic_df.shape)

## 3. 🔍 Exploratory Data Analysis

### 3.1 Missing Values

In [ ]:
print("Missing values per column:")
print(titanic_df.isnull().sum())
print(f"\nTotal missing values: {titanic_df.isnull().sum().sum()}")

### 3.2 Descriptive Statistics

In [ ]:
titanic_df.describe()

In [ ]:
titanic_df.info()

### 3.3 Target Distribution

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(titanic_df["Survived"], bins=2, edgecolor="black", color="red")
plt.title("Histogram of Survived")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.xticks([0, 1])
plt.tight_layout()
plt.show()

## 4. 🧹 Data Preprocessing

### 4.1 Drop Cabin Column
> `Cabin` has too many missing values (~77%) to be reliably imputed — dropping it.

In [ ]:
titanic_clean = titanic_df.copy()
titanic_clean = titanic_clean.drop("Cabin", axis=1)
titanic_clean.shape

### 4.2 Encode Sex
> Map `female → 0`, `male → 1`

In [ ]:
titanic_clean['Sex'] = (
    titanic_clean['Sex'].astype(str).str.strip().str.lower()
    .map({'female': 0, 'male': 1})
    .fillna(-1)
    .astype(int)
)
titanic_clean['Sex'].value_counts()

### 4.3 Fill Missing Ages
> Replace missing `Age` values with the **median** age.

In [ ]:
titanic_clean["Age"] = titanic_clean["Age"].fillna(titanic_clean["Age"].median()).astype(int)
print("Remaining Age nulls:", titanic_clean["Age"].isnull().sum())

### 4.4 Fill Missing Embarked

In [ ]:
titanic_clean["Embarked"] = titanic_clean["Embarked"].fillna('C')
titanic_clean["Embarked"].value_counts()

### 4.5 Binarize Fare
> Split `Fare` into **cheap (0)** vs **expensive (1)** based on the median.

In [ ]:
med_fare = titanic_clean["Fare"].median()
print(f"Median Fare: {med_fare}")

titanic_clean["Fare"] = (titanic_clean["Fare"] > med_fare).astype(int)
titanic_clean["Fare"].value_counts()

### 4.6 Create FamilySize Feature

In [ ]:
titanic_clean["FamilySize"] = titanic_clean["SibSp"] + titanic_clean["Parch"] + 1
titanic_clean["FamilySize"].value_counts().sort_index()

### 4.7 Encode Ticket Prefix

In [ ]:
titanic_clean = titanic_clean.drop("Name", axis=1)
titanic_clean["TicketPrefix"] = titanic_clean["Ticket"].str.extract(r"^([A-Za-z/.]+)")
titanic_clean["TicketPrefix"] = titanic_clean["TicketPrefix"].fillna("NONE")

encoder = LabelEncoder()
titanic_clean["TicketPrefix_encoded"] = encoder.fit_transform(titanic_clean["TicketPrefix"])
titanic_clean.drop(["TicketPrefix", "Ticket"], axis=1, inplace=True)
titanic_clean["TicketPrefix_encoded"].value_counts().head()

### 4.8 Encode Embarked

In [ ]:
titanic_clean["Embarked_encoded"] = encoder.fit_transform(titanic_clean["Embarked"])
titanic_clean.drop("Embarked", axis=1, inplace=True)
titanic_clean.head()

### 4.9 Cleaned Dataset Overview

In [ ]:
print("Final shape:", titanic_clean.shape)
titanic_clean.dtypes

## 5. 🎯 Feature Selection & Correlation Analysis

In [ ]:
feature_columns = [
    "PassengerId", "Pclass", "Sex", "Age",
    "Fare", "FamilySize", "TicketPrefix_encoded", "Embarked_encoded"
]

X = titanic_clean[feature_columns]
y = titanic_clean["Survived"]

print("Features shape:", X.shape)
print("Target shape:  ", y.shape)

In [ ]:
corr_data = titanic_clean[feature_columns + ["Survived"]]
corr_mat = corr_data.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap="coolwarm", center=0, linewidths=.5)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

print("\nCorrelation with Survived:")
print(corr_mat['Survived'].sort_values(ascending=False))

## 6. ✂️ Train/Test Split & Feature Scaling

In [ ]:
np.random.seed(42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set : {X_train.shape[0]} samples")
print(f"Testing set  : {X_test.shape[0]} samples")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\n✅ Features scaled!")
print("Before scaling (first sample):", X_train.iloc[0].values)
print("After  scaling (first sample):", X_train_scaled[0].round(3))

## 7. 📈 Logistic Regression

In [ ]:
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)

y_pred_train = logreg.predict(X_train_scaled)
y_pred_test  = logreg.predict(X_test_scaled)
y_pred_proba = logreg.predict_proba(X_test_scaled)[:, 1]

### 7.1 Performance Metrics

In [ ]:
def print_metrics(y_true, y_pred, label):
    print(f"--- {label} ---")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score : {f1_score(y_true, y_pred):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print()

print_metrics(y_train, y_pred_train, "Logistic Regression — Training Set")
print_metrics(y_test,  y_pred_test,  "Logistic Regression — Test Set")
print(f"ROC AUC (Test): {roc_auc_score(y_test, y_pred_proba):.4f}")

### 7.2 ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
auc_score = roc_auc_score(y_test, y_pred_proba)

plt.figure(figsize=(7, 5))
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc_score:.3f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Logistic Regression — ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

## 8. 🔵 K-Nearest Neighbors (KNN)

In [ ]:
neighbors = np.arange(1, 26)
train_accuracies = {}
test_accuracies  = {}

for k in neighbors:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_accuracies[k] = knn.score(X_train_scaled, y_train)
    test_accuracies[k]  = knn.score(X_test_scaled,  y_test)

# Keep best KNN model (k=12)
best_k = max(test_accuracies, key=test_accuracies.get)
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_scaled, y_train)
print(f"Best k (by test accuracy): {best_k} → {test_accuracies[best_k]:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.title("KNN: Varying Number of Neighbors")
plt.plot(neighbors, train_accuracies.values(), label="Training Accuracy", marker='o', markersize=4)
plt.plot(neighbors, test_accuracies.values(),  label="Testing Accuracy",  marker='s', markersize=4)
plt.axvline(best_k, color='red', linestyle='--', alpha=0.5, label=f'Best k={best_k}')
plt.legend()
plt.xlabel("Number of Neighbors")
plt.ylabel("Accuracy")
plt.tight_layout()
plt.show()

## 9. 🔮 Predict Survival for New Passengers

In [ ]:
def predict_survival(pclass, sex, age, fare, family_size, ticket_prefix_encoded, embarked_encoded):
    """
    Predict Titanic survival using Logistic Regression and KNN.

    Parameters
    ----------
    pclass                : int  — Passenger class (1, 2, 3)
    sex                   : int  — 0 = female, 1 = male
    age                   : int  — Passenger age
    fare                  : int  — 0 = cheap, 1 = expensive
    family_size           : int  — SibSp + Parch + 1
    ticket_prefix_encoded : int  — Encoded ticket prefix
    embarked_encoded      : int  — Encoded port of embarkation

    Returns
    -------
    dict with predictions from both models
    """
    df = pd.DataFrame([{
        "PassengerId"         : 0,
        "Pclass"              : pclass,
        "Sex"                 : sex,
        "Age"                 : age,
        "Fare"                : fare,
        "FamilySize"          : family_size,
        "TicketPrefix_encoded": ticket_prefix_encoded,
        "Embarked_encoded"    : embarked_encoded
    }])
    df_scaled = scaler.transform(df)

    pred_log  = logreg.predict(df_scaled)[0]
    prob_log  = round(float(logreg.predict_proba(df_scaled)[:, 1][0]), 3)
    pred_knn  = knn.predict(df_scaled)[0]

    return {
        "Logistic Regression": "Survived ✅" if pred_log == 1 else "Did Not Survive ❌",
        "LR Probability"     : prob_log,
        "KNN"                : "Survived ✅" if pred_knn == 1 else "Did Not Survive ❌"
    }

### Examples

In [ ]:
# Example 1: 29-year-old male, 3rd class, cheap ticket, travelling alone
e1 = predict_survival(pclass=3, sex=1, age=29, fare=0, family_size=1,
                       ticket_prefix_encoded=0, embarked_encoded=2)
print("Example 1 — 29-year-old male, 3rd class:")
for k, v in e1.items(): print(f"  {k}: {v}")

print()

# Example 2: 35-year-old female, 1st class, expensive ticket, family of 2
e2 = predict_survival(pclass=1, sex=0, age=35, fare=1, family_size=2,
                       ticket_prefix_encoded=5, embarked_encoded=0)
print("Example 2 — 35-year-old female, 1st class:")
for k, v in e2.items(): print(f"  {k}: {v}")

print()

# Example 3: 22-year-old male, 2nd class, cheap ticket, travelling alone
e3 = predict_survival(pclass=2, sex=1, age=22, fare=0, family_size=1,
                       ticket_prefix_encoded=3, embarked_encoded=1)
print("Example 3 — 22-year-old male, 2nd class:")
for k, v in e3.items(): print(f"  {k}: {v}")